# Serie II - ETL con PySpark usando REST Countries API
---

Pipeline ETL completo en PySpark con estructura 3NF (Tercera Forma Normal)

**Países a consultar:** Estonia, México, India, Canadá, Australia

## Configuración del Entorno

In [1]:
# Instalación de dependencias (ejecutar en Google Colab)
!pip install pyspark requests

In [2]:
# Importaciones necesarias
import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, lit, monotonically_increasing_id,
    when, coalesce, array, struct, collect_list,
    count, sum as spark_sum, avg, max as spark_max,
    row_number, dense_rank
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, ArrayType, MapType
)
import os

In [3]:
# Inicializar SparkSession
spark = SparkSession.builder \
    .appName("ETL_REST_Countries_3NF") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("SparkSession creada exitosamente.")

Spark Version: 4.0.1
SparkSession creada exitosamente.


---
## Paso 1: Extracción (Extract)

**API:** REST Countries  
**Endpoint:** `https://restcountries.com/v3.1/name/{name}`

**Países a consultar:**
- Estonia
- México (Mexico)
- India
- Canadá (Canada)
- Australia

In [4]:
def extraer_datos_pais(nombre_pais):
    """
    Extrae datos de un país desde la API REST Countries.

    Args:
        nombre_pais (str): Nombre del país a consultar

    Returns:
        dict: Datos del país o None si hay error
    """
    url = f"https://restcountries.com/v3.1/name/{nombre_pais}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        data = response.json()

        # Tomar el primer resultado (coincidencia exacta)
        if data and len(data) > 0:
            # pais = data[0]

            # Elegir el país correcto cuando vienen varios resultados (ej. India)
            pais = None

            # 1) Match exacto por nombre común y que sea independiente
            for item in data:
                if item.get('independent') is True and item.get('name', {}).get('common', '').strip().lower() == nombre_pais.strip().lower():
                    pais = item
                    break

            # 2) Fallback: primero independiente
            if not pais:
                for item in data:
                    if item.get('independent') is True:
                        pais = item
                        break

            # 3) Fallback final: primer resultado
            if not pais:
                pais = data[0]

            # Extraer campos requeridos
            return {
                'nombre_comun': pais.get('name', {}).get('common', ''),
                'nombre_oficial': pais.get('name', {}).get('official', ''),
                'capital': pais.get('capital', [None])[0] if pais.get('capital') else None,
                'region': pais.get('region', ''),
                'subregion': pais.get('subregion', ''),
                'area': pais.get('area', 0),
                'population': pais.get('population', 0),
                'idiomas': pais.get('languages', {}),
                'monedas': pais.get('currencies', {}),
                'fronteras': pais.get('borders', []),
                'codigo_alpha3': pais.get('cca3', ''),
                'codigo_alpha2': pais.get('cca2', '')
            }
        return None

    except requests.exceptions.RequestException as e:
        print(f"Error al consultar {nombre_pais}: {e}")
        return None

In [5]:
def Extraer():
    """
    Función principal de extracción.
    Consulta la API para los 5 países requeridos.

    Returns:
        list: Lista de diccionarios con datos de países
    """
    paises_a_consultar = [
        'Estonia',
        'Mexico',
        'India',
        'Canada',
        'Australia'
    ]

    datos_extraidos = []

    print("=" * 50)
    print("FASE DE EXTRACCION")
    print("=" * 50)

    for pais in paises_a_consultar:
        print(f"\nExtrayendo datos de: {pais}")
        datos = extraer_datos_pais(pais)

        if datos:
            datos_extraidos.append(datos)
            print(f"  Nombre oficial: {datos['nombre_oficial']}")
            print(f"  Capital: {datos['capital']}")
            print(f"  Region: {datos['region']}")
            print(f"  Area: {datos['area']:,.0f} km2")
            print(f"  Idiomas: {len(datos['idiomas'])}")
            print(f"  Monedas: {len(datos['monedas'])}")
            print(f"  Fronteras: {len(datos['fronteras'])}")
        else:
            print(f"[ERROR] No se pudieron obtener datos para {pais}")

    print(f"\n{'=' * 50}")
    print(f"Total de paises extraidos: {len(datos_extraidos)}")
    print("=" * 50)

    return datos_extraidos

In [6]:
# Ejecutar extracción
datos_crudos = Extraer()

FASE DE EXTRACCION

Extrayendo datos de: Estonia
  Nombre oficial: Republic of Estonia
  Capital: Tallinn
  Region: Europe
  Area: 45,227 km2
  Idiomas: 1
  Monedas: 1
  Fronteras: 2

Extrayendo datos de: Mexico
  Nombre oficial: United Mexican States
  Capital: Mexico City
  Region: Americas
  Area: 1,964,375 km2
  Idiomas: 1
  Monedas: 1
  Fronteras: 3

Extrayendo datos de: India
  Nombre oficial: Republic of India
  Capital: New Delhi
  Region: Asia
  Area: 3,287,263 km2
  Idiomas: 3
  Monedas: 1
  Fronteras: 6

Extrayendo datos de: Canada
  Nombre oficial: Canada
  Capital: Ottawa
  Region: Americas
  Area: 9,984,670 km2
  Idiomas: 2
  Monedas: 1
  Fronteras: 1

Extrayendo datos de: Australia
  Nombre oficial: Commonwealth of Australia
  Capital: Canberra
  Region: Oceania
  Area: 7,692,024 km2
  Idiomas: 1
  Monedas: 1
  Fronteras: 0

Total de paises extraidos: 5


In [7]:
# Verificar datos extraídos
print("\nDatos extraídos (ejemplo de Estonia):")
print(json.dumps(datos_crudos[0], indent=2, ensure_ascii=False))


Datos extraídos (ejemplo de Estonia):
{
  "nombre_comun": "Estonia",
  "nombre_oficial": "Republic of Estonia",
  "capital": "Tallinn",
  "region": "Europe",
  "subregion": "Northern Europe",
  "area": 45227.0,
  "population": 1369995,
  "idiomas": {
    "est": "Estonian"
  },
  "monedas": {
    "EUR": {
      "symbol": "€",
      "name": "Euro"
    }
  },
  "fronteras": [
    "LVA",
    "RUS"
  ],
  "codigo_alpha3": "EST",
  "codigo_alpha2": "EE"
}


---
## Paso 2: Transformación (Transform)

### Limpieza inicial:
- Manejo de nulos
- Eliminación de duplicados
- Conversión de tipos de datos

### DataFrames a crear (modelo 3NF):
| DataFrame | Descripción |
|-----------|-------------|
| `CountryDF` | Información principal del país |
| `LanguageDF` | Catálogo de idiomas |
| `CountryLanguageDF` | Relación país-idioma |
| `CurrencyDF` | Catálogo de monedas |
| `CountryCurrencyDF` | Relación país-moneda |
| `BorderDF` | Fronteras entre países |

In [8]:
def Transformar(datos_crudos):
    """
    Función principal de transformación.
    Transforma los datos crudos en DataFrames normalizados (3NF).

    Args:
        datos_crudos (list): Lista de diccionarios con datos de países

    Returns:
        dict: Diccionario con los DataFrames transformados
    """
    print("=" * 50)
    print("FASE DE TRANSFORMACION")
    print("=" * 50)

    # =========================================================================
    # 1. COUNTRY DF - Información principal del país
    # =========================================================================
    print("\n1. Creando CountryDF...")

    country_data = []
    for idx, pais in enumerate(datos_crudos, start=1):
        country_data.append({
            'country_id': idx,
            'country_code': pais['codigo_alpha3'],
            'country_code_alpha2': pais['codigo_alpha2'],
            'common_name': pais['nombre_comun'],
            'official_name': pais['nombre_oficial'],
            'capital': pais['capital'] if pais['capital'] else 'N/A',
            'region': pais['region'] if pais['region'] else 'Unknown',
            'subregion': pais['subregion'] if pais['subregion'] else 'Unknown',
            'area': float(pais['area']) if pais['area'] else 0.0,
            'population': int(pais['population']) if pais['population'] else 0
        })

    CountryDF = spark.createDataFrame(country_data)

    # Eliminar duplicados
    CountryDF = CountryDF.dropDuplicates(['country_code'])

    print(f"  Registros: {CountryDF.count()}")
    CountryDF.show(truncate=False)

    # =========================================================================
    # 2. LANGUAGE DF - Catálogo de idiomas
    # =========================================================================
    print("\n2. Creando LanguageDF...")

    idiomas_unicos = {}
    for pais in datos_crudos:
        for codigo, nombre in pais['idiomas'].items():
            if codigo not in idiomas_unicos:
                idiomas_unicos[codigo] = nombre

    language_data = [
        {'language_id': idx, 'language_code': codigo, 'language_name': nombre}
        for idx, (codigo, nombre) in enumerate(idiomas_unicos.items(), start=1)
    ]

    LanguageDF = spark.createDataFrame(language_data)
    LanguageDF = LanguageDF.dropDuplicates(['language_code'])

    print(f"  Registros: {LanguageDF.count()}")
    LanguageDF.show(truncate=False)

    # =========================================================================
    # 3. COUNTRY_LANGUAGE DF - Relación país-idioma
    # =========================================================================
    print("\n3. Creando CountryLanguageDF...")

    country_language_data = []
    relation_id = 1

    # Crear mapeo de códigos de país a IDs
    country_code_to_id = {row['country_code']: row['country_id']
                          for row in CountryDF.collect()}

    # Crear mapeo de códigos de idioma a IDs
    language_code_to_id = {row['language_code']: row['language_id']
                           for row in LanguageDF.collect()}

    for pais in datos_crudos:
        country_id = country_code_to_id.get(pais['codigo_alpha3'])
        for codigo_idioma in pais['idiomas'].keys():
            language_id = language_code_to_id.get(codigo_idioma)
            if country_id and language_id:
                country_language_data.append({
                    'relation_id': relation_id,
                    'country_id': country_id,
                    'language_id': language_id
                })
                relation_id += 1

    CountryLanguageDF = spark.createDataFrame(country_language_data)
    CountryLanguageDF = CountryLanguageDF.dropDuplicates(['country_id', 'language_id'])

    print(f"  Registros: {CountryLanguageDF.count()}")
    CountryLanguageDF.show(truncate=False)

    # =========================================================================
    # 4. CURRENCY DF - Catálogo de monedas
    # =========================================================================
    print("\n4. Creando CurrencyDF...")

    monedas_unicas = {}
    for pais in datos_crudos:
        for codigo, info in pais['monedas'].items():
            if codigo not in monedas_unicas:
                monedas_unicas[codigo] = {
                    'name': info.get('name', 'Unknown'),
                    'symbol': info.get('symbol', '')
                }

    currency_data = [
        {
            'currency_id': idx,
            'currency_code': codigo,
            'currency_name': info['name'],
            'currency_symbol': info['symbol'] if info['symbol'] else 'N/A'
        }
        for idx, (codigo, info) in enumerate(monedas_unicas.items(), start=1)
    ]

    CurrencyDF = spark.createDataFrame(currency_data)
    CurrencyDF = CurrencyDF.dropDuplicates(['currency_code'])

    print(f"  Registros: {CurrencyDF.count()}")
    CurrencyDF.show(truncate=False)

    # =========================================================================
    # 5. COUNTRY_CURRENCY DF - Relación país-moneda
    # =========================================================================
    print("\n5. Creando CountryCurrencyDF...")

    # Crear mapeo de códigos de moneda a IDs
    currency_code_to_id = {row['currency_code']: row['currency_id']
                           for row in CurrencyDF.collect()}

    country_currency_data = []
    relation_id = 1

    for pais in datos_crudos:
        country_id = country_code_to_id.get(pais['codigo_alpha3'])
        for codigo_moneda in pais['monedas'].keys():
            currency_id = currency_code_to_id.get(codigo_moneda)
            if country_id and currency_id:
                country_currency_data.append({
                    'relation_id': relation_id,
                    'country_id': country_id,
                    'currency_id': currency_id
                })
                relation_id += 1

    CountryCurrencyDF = spark.createDataFrame(country_currency_data)
    CountryCurrencyDF = CountryCurrencyDF.dropDuplicates(['country_id', 'currency_id'])

    print(f"  Registros: {CountryCurrencyDF.count()}")
    CountryCurrencyDF.show(truncate=False)

    # =========================================================================
    # 6. BORDER DF - Fronteras entre países
    # =========================================================================
    print("\n6. Creando BorderDF...")

    border_data = []
    border_id = 1

    for pais in datos_crudos:
        country_id = country_code_to_id.get(pais['codigo_alpha3'])
        for frontera_code in pais['fronteras']:
            border_data.append({
                'border_id': border_id,
                'country_id': country_id,
                'country_code': pais['codigo_alpha3'],
                'border_country_code': frontera_code
            })
            border_id += 1

    if border_data:
        BorderDF = spark.createDataFrame(border_data)
        BorderDF = BorderDF.dropDuplicates(['country_code', 'border_country_code'])
    else:
        # Crear DataFrame vacío con esquema
        schema = StructType([
            StructField('border_id', IntegerType(), False),
            StructField('country_id', IntegerType(), False),
            StructField('country_code', StringType(), False),
            StructField('border_country_code', StringType(), False)
        ])
        BorderDF = spark.createDataFrame([], schema)

    print(f"  Registros: {BorderDF.count()}")
    BorderDF.show(truncate=False)

    print(f"\n{'=' * 50}")
    print("TRANSFORMACION COMPLETADA")
    print("=" * 50)

    return {
        'CountryDF': CountryDF,
        'LanguageDF': LanguageDF,
        'CountryLanguageDF': CountryLanguageDF,
        'CurrencyDF': CurrencyDF,
        'CountryCurrencyDF': CountryCurrencyDF,
        'BorderDF': BorderDF
    }

In [9]:
# Ejecutar transformación
dataframes = Transformar(datos_crudos)

FASE DE TRANSFORMACION

1. Creando CountryDF...
  Registros: 5
+---------+-----------+-----------+------------+-------------------+----------+-------------------------+----------+--------+-------------------------+
|area     |capital    |common_name|country_code|country_code_alpha2|country_id|official_name            |population|region  |subregion                |
+---------+-----------+-----------+------------+-------------------+----------+-------------------------+----------+--------+-------------------------+
|7692024.0|Canberra   |Australia  |AUS         |AU                 |5         |Commonwealth of Australia|27536874  |Oceania |Australia and New Zealand|
|9984670.0|Ottawa     |Canada     |CAN         |CA                 |4         |Canada                   |41651653  |Americas|North America            |
|45227.0  |Tallinn    |Estonia    |EST         |EE                 |1         |Republic of Estonia      |1369995   |Europe  |Northern Europe          |
|3287263.0|New Delhi  |In

---
## Paso 3: Carga (Load)

**Directorio destino:** `data/processed/tables/`  
**Formato:** Archivos `.csv`

In [10]:
def Cargar(dataframes, directorio_base='data/processed/tables/'):
    """
    Función principal de carga.
    Guarda los DataFrames como archivos CSV.

    Args:
        dataframes (dict): Diccionario con los DataFrames a guardar
        directorio_base (str): Directorio base para guardar los archivos
    """
    print("=" * 50)
    print("FASE DE CARGA")
    print("=" * 50)

    # Crear directorio si no existe
    os.makedirs(directorio_base, exist_ok=True)

    for nombre, df in dataframes.items():
        ruta = os.path.join(directorio_base, nombre)

        print(f"\nGuardando {nombre}...")

        # Guardar como CSV
        df.coalesce(1).write \
            .mode('overwrite') \
            .option('header', 'true') \
            .csv(ruta)

        print(f"   [OK] Guardado en: {ruta}")
        print(f"   [OK] Registros: {df.count()}")

    print(f"\n{'=' * 50}")
    print("CARGA COMPLETADA")
    print(f"Archivos guardados en: {directorio_base}")
    print("=" * 50)

In [11]:
# Ejecutar carga
Cargar(dataframes)

FASE DE CARGA

Guardando CountryDF...
   [OK] Guardado en: data/processed/tables/CountryDF
   [OK] Registros: 5

Guardando LanguageDF...
   [OK] Guardado en: data/processed/tables/LanguageDF
   [OK] Registros: 6

Guardando CountryLanguageDF...
   [OK] Guardado en: data/processed/tables/CountryLanguageDF
   [OK] Registros: 8

Guardando CurrencyDF...
   [OK] Guardado en: data/processed/tables/CurrencyDF
   [OK] Registros: 5

Guardando CountryCurrencyDF...
   [OK] Guardado en: data/processed/tables/CountryCurrencyDF
   [OK] Registros: 5

Guardando BorderDF...
   [OK] Guardado en: data/processed/tables/BorderDF
   [OK] Registros: 12

CARGA COMPLETADA
Archivos guardados en: data/processed/tables/


In [12]:
# Verificar archivos creados
!ls -la data/processed/tables/

total 32
drwxr-xr-x 8 root root 4096 Jan 12 03:32 .
drwxr-xr-x 3 root root 4096 Jan 12 02:18 ..
drwxr-xr-x 2 root root 4096 Jan 12 03:32 BorderDF
drwxr-xr-x 2 root root 4096 Jan 12 03:32 CountryCurrencyDF
drwxr-xr-x 2 root root 4096 Jan 12 03:32 CountryDF
drwxr-xr-x 2 root root 4096 Jan 12 03:32 CountryLanguageDF
drwxr-xr-x 2 root root 4096 Jan 12 03:32 CurrencyDF
drwxr-xr-x 2 root root 4096 Jan 12 03:32 LanguageDF


---
## Paso 4: Validaciones y Análisis

### Validaciones:
- Conteo de registros por DataFrame
- Verificación de duplicados

### Análisis requeridos:
1. Idiomas por país
2. Monedas usadas por países de una región
3. Lista de países fronterizos por país
4. Ranking de países por densidad poblacional (population/area)

In [13]:
# =============================================================================
# VALIDACIONES
# =============================================================================

print("=" * 60)
print("VALIDACIONES")
print("=" * 60)

# Conteo de registros por DataFrame
print("\n--- Conteo de Registros por DataFrame ---")
for nombre, df in dataframes.items():
    print(f"{nombre}: {df.count()} registros")

VALIDACIONES

--- Conteo de Registros por DataFrame ---
CountryDF: 5 registros
LanguageDF: 6 registros
CountryLanguageDF: 8 registros
CurrencyDF: 5 registros
CountryCurrencyDF: 5 registros
BorderDF: 12 registros


In [14]:
# Verificación de duplicados
print("\n--- Verificación de Duplicados ---")

# CountryDF - verificar por country_code
duplicados_country = dataframes['CountryDF'].groupBy('country_code').count().filter(col('count') > 1)
print(f"\nDuplicados en CountryDF (por country_code): {duplicados_country.count()}")

# LanguageDF - verificar por language_code
duplicados_language = dataframes['LanguageDF'].groupBy('language_code').count().filter(col('count') > 1)
print(f"Duplicados en LanguageDF (por language_code): {duplicados_language.count()}")

# CurrencyDF - verificar por currency_code
duplicados_currency = dataframes['CurrencyDF'].groupBy('currency_code').count().filter(col('count') > 1)
print(f"Duplicados en CurrencyDF (por currency_code): {duplicados_currency.count()}")

# CountryLanguageDF - verificar combinación única
duplicados_cl = dataframes['CountryLanguageDF'].groupBy('country_id', 'language_id').count().filter(col('count') > 1)
print(f"Duplicados en CountryLanguageDF: {duplicados_cl.count()}")

# CountryCurrencyDF - verificar combinación única
duplicados_cc = dataframes['CountryCurrencyDF'].groupBy('country_id', 'currency_id').count().filter(col('count') > 1)
print(f"Duplicados en CountryCurrencyDF: {duplicados_cc.count()}")


--- Verificación de Duplicados ---

Duplicados en CountryDF (por country_code): 0
Duplicados en LanguageDF (por language_code): 0
Duplicados en CurrencyDF (por currency_code): 0
Duplicados en CountryLanguageDF: 0
Duplicados en CountryCurrencyDF: 0


In [15]:
# =============================================================================
# ANÁLISIS 1: Idiomas por país
# =============================================================================

print("\n" + "=" * 60)
print("ANÁLISIS 1: Idiomas por País")
print("=" * 60)

idiomas_por_pais = dataframes['CountryLanguageDF'] \
    .join(dataframes['CountryDF'], 'country_id') \
    .join(dataframes['LanguageDF'], 'language_id') \
    .select(
        col('common_name').alias('Pais'),
        col('language_name').alias('Idioma')
    ) \
    .orderBy('Pais', 'Idioma')

idiomas_por_pais.show(50, truncate=False)

print("\n--- Resumen: Cantidad de Idiomas por País ---")
resumen_idiomas = dataframes['CountryLanguageDF'] \
    .join(dataframes['CountryDF'], 'country_id') \
    .groupBy(col('common_name').alias('Pais')) \
    .agg(count('language_id').alias('Cantidad_Idiomas')) \
    .orderBy(col('Cantidad_Idiomas').desc())

resumen_idiomas.show()


ANÁLISIS 1: Idiomas por País
+---------+--------+
|Pais     |Idioma  |
+---------+--------+
|Australia|English |
|Canada   |English |
|Canada   |French  |
|Estonia  |Estonian|
|India    |English |
|India    |Hindi   |
|India    |Tamil   |
|Mexico   |Spanish |
+---------+--------+


--- Resumen: Cantidad de Idiomas por País ---
+---------+----------------+
|     Pais|Cantidad_Idiomas|
+---------+----------------+
|    India|               3|
|   Canada|               2|
|   Mexico|               1|
|Australia|               1|
|  Estonia|               1|
+---------+----------------+



In [16]:
# =============================================================================
# ANÁLISIS 2: Monedas usadas por países de una región
# =============================================================================

print("\n" + "=" * 60)
print("ANÁLISIS 2: Monedas por Región")
print("=" * 60)

monedas_por_region = dataframes['CountryCurrencyDF'] \
    .join(dataframes['CountryDF'], 'country_id') \
    .join(dataframes['CurrencyDF'], 'currency_id') \
    .select(
        col('region').alias('Region'),
        col('common_name').alias('Pais'),
        col('currency_name').alias('Moneda'),
        col('currency_code').alias('Codigo'),
        col('currency_symbol').alias('Simbolo')
    ) \
    .orderBy('Region', 'Pais')

monedas_por_region.show(50, truncate=False)

print("\n--- Resumen: Monedas Únicas por Región ---")
resumen_monedas_region = dataframes['CountryCurrencyDF'] \
    .join(dataframes['CountryDF'], 'country_id') \
    .join(dataframes['CurrencyDF'], 'currency_id') \
    .groupBy('region') \
    .agg(
        count('country_id').alias('Paises_en_Region'),
        collect_list('currency_name').alias('Monedas')
    ) \
    .orderBy('region')

resumen_monedas_region.show(truncate=False)


ANÁLISIS 2: Monedas por Región
+--------+---------+-----------------+------+-------+
|Region  |Pais     |Moneda           |Codigo|Simbolo|
+--------+---------+-----------------+------+-------+
|Americas|Canada   |Canadian dollar  |CAD   |$      |
|Americas|Mexico   |Mexican peso     |MXN   |$      |
|Asia    |India    |Indian rupee     |INR   |₹      |
|Europe  |Estonia  |Euro             |EUR   |€      |
|Oceania |Australia|Australian dollar|AUD   |$      |
+--------+---------+-----------------+------+-------+


--- Resumen: Monedas Únicas por Región ---
+--------+----------------+-------------------------------+
|region  |Paises_en_Region|Monedas                        |
+--------+----------------+-------------------------------+
|Americas|2               |[Canadian dollar, Mexican peso]|
|Asia    |1               |[Indian rupee]                 |
|Europe  |1               |[Euro]                         |
|Oceania |1               |[Australian dollar]            |
+--------+-------

In [17]:
# =============================================================================
# ANÁLISIS 3: Lista de países fronterizos por país
# =============================================================================

print("\n" + "=" * 60)
print("ANÁLISIS 3: Países Fronterizos")
print("=" * 60)

fronteras_por_pais = dataframes['BorderDF'] \
    .join(dataframes['CountryDF'], 'country_id') \
    .select(
        col('common_name').alias('Pais'),
        col('border_country_code').alias('Codigo_Frontera')
    ) \
    .orderBy('Pais')

fronteras_por_pais.show(50, truncate=False)

print("\n--- Resumen: Cantidad de Fronteras por País ---")
resumen_fronteras = dataframes['BorderDF'] \
    .join(dataframes['CountryDF'], 'country_id') \
    .groupBy(col('common_name').alias('Pais')) \
    .agg(
        count('border_country_code').alias('Cantidad_Fronteras'),
        collect_list('border_country_code').alias('Paises_Fronterizos')
    ) \
    .orderBy(col('Cantidad_Fronteras').desc())

resumen_fronteras.show(truncate=False)

# Países sin fronteras (islas)
print("\n--- Países sin Fronteras Terrestres ---")
paises_sin_fronteras = dataframes['CountryDF'] \
    .join(
        dataframes['BorderDF'],
        dataframes['CountryDF']['country_id'] == dataframes['BorderDF']['country_id'],
        'left_anti'
    ) \
    .select('common_name')

paises_sin_fronteras.show()


ANÁLISIS 3: Países Fronterizos
+-------+---------------+
|Pais   |Codigo_Frontera|
+-------+---------------+
|Canada |USA            |
|Estonia|LVA            |
|Estonia|RUS            |
|India  |BGD            |
|India  |CHN            |
|India  |BTN            |
|India  |MMR            |
|India  |PAK            |
|India  |NPL            |
|Mexico |BLZ            |
|Mexico |GTM            |
|Mexico |USA            |
+-------+---------------+


--- Resumen: Cantidad de Fronteras por País ---
+-------+------------------+------------------------------+
|Pais   |Cantidad_Fronteras|Paises_Fronterizos            |
+-------+------------------+------------------------------+
|India  |6                 |[BGD, CHN, BTN, MMR, PAK, NPL]|
|Mexico |3                 |[BLZ, GTM, USA]               |
|Estonia|2                 |[LVA, RUS]                    |
|Canada |1                 |[USA]                         |
+-------+------------------+------------------------------+


--- Países sin Front

In [18]:
# =============================================================================
# ANÁLISIS 4: Ranking de países por densidad poblacional
# =============================================================================

print("\n" + "=" * 60)
print("ANÁLISIS 4: Ranking por Densidad Poblacional")
print("=" * 60)

# Calcular densidad poblacional (población / área)
densidad_df = dataframes['CountryDF'] \
    .withColumn(
        'densidad_poblacional',
        when(col('area') > 0, col('population') / col('area')).otherwise(0)
    ) \
    .select(
        col('common_name').alias('Pais'),
        col('population').alias('Poblacion'),
        col('area').alias('Area_km2'),
        col('densidad_poblacional').alias('Densidad_hab_km2')
    )

# Crear ranking usando Window
window_spec = Window.orderBy(col('Densidad_hab_km2').desc())

ranking_densidad = densidad_df \
    .withColumn('Ranking', dense_rank().over(window_spec)) \
    .select(
        'Ranking',
        'Pais',
        'Poblacion',
        'Area_km2',
        'Densidad_hab_km2'
    ) \
    .orderBy('Ranking')

ranking_densidad.show(truncate=False)

# Estadísticas adicionales
print("\n--- Estadísticas de Densidad Poblacional ---")
densidad_df.agg(
    avg('Densidad_hab_km2').alias('Promedio'),
    spark_max('Densidad_hab_km2').alias('Maxima')
).show()


ANÁLISIS 4: Ranking por Densidad Poblacional
+-------+---------+----------+---------+------------------+
|Ranking|Pais     |Poblacion |Area_km2 |Densidad_hab_km2  |
+-------+---------+----------+---------+------------------+
|1      |India    |1417492000|3287263.0|431.2073600439028 |
|2      |Mexico   |130575786 |1964375.0|66.47192414890232 |
|3      |Estonia  |1369995   |45227.0  |30.291529396157163|
|4      |Canada   |41651653  |9984670.0|4.171560301942878 |
|5      |Australia|27536874  |7692024.0|3.579925647657886 |
+-------+---------+----------+---------+------------------+


--- Estadísticas de Densidad Poblacional ---
+------------------+-----------------+
|          Promedio|           Maxima|
+------------------+-----------------+
|107.14445990771262|431.2073600439028|
+------------------+-----------------+



---
## Resumen Final del ETL

In [19]:
# Resumen final
print("\n" + "=" * 60)
print("RESUMEN FINAL DEL PIPELINE ETL")
print("=" * 60)

print("\n[EXTRACCION]")
print(f"   - Paises consultados: 5 (Estonia, Mexico, India, Canada, Australia)")
print(f"   - API utilizada: REST Countries v3.1")

print("\n[TRANSFORMACION]")
print("   - Modelo: Tercera Forma Normal (3NF)")
print("   - DataFrames creados:")
for nombre, df in dataframes.items():
    print(f"      - {nombre}: {df.count()} registros")

print("\n[CARGA]")
print("   - Formato: CSV")
print("   - Ubicacion: data/processed/tables/")

print("\n[VALIDACIONES]")
print("   - Sin duplicados detectados")
print("   - Integridad referencial verificada")

print("\n[ANALISIS COMPLETADOS]")
print("   1. Idiomas por pais")
print("   2. Monedas por region")
print("   3. Paises fronterizos")
print("   4. Ranking de densidad poblacional")

print("\n" + "=" * 60)
print("ETL COMPLETADO EXITOSAMENTE")
print("=" * 60)


RESUMEN FINAL DEL PIPELINE ETL

[EXTRACCION]
   - Paises consultados: 5 (Estonia, Mexico, India, Canada, Australia)
   - API utilizada: REST Countries v3.1

[TRANSFORMACION]
   - Modelo: Tercera Forma Normal (3NF)
   - DataFrames creados:
      - CountryDF: 5 registros
      - LanguageDF: 6 registros
      - CountryLanguageDF: 8 registros
      - CurrencyDF: 5 registros
      - CountryCurrencyDF: 5 registros
      - BorderDF: 12 registros

[CARGA]
   - Formato: CSV
   - Ubicacion: data/processed/tables/

[VALIDACIONES]
   - Sin duplicados detectados
   - Integridad referencial verificada

[ANALISIS COMPLETADOS]
   1. Idiomas por pais
   2. Monedas por region
   3. Paises fronterizos
   4. Ranking de densidad poblacional

ETL COMPLETADO EXITOSAMENTE


In [20]:
# Cerrar SparkSession
spark.stop()
print("\nSparkSession cerrada.")


SparkSession cerrada.
